# FlashNystrom — Colab experiments → paper artifacts (saved to Drive)

Two experiments, both saved to Google Drive (JSON + logs + PDF/PNG figures + zip):
1. **Scaling / crossover** — throughput & peak memory vs sequence length (where FN overtakes full attention and sdpa OOMs).
2. **STL-10** — a real image classifier at large token counts (96x96 native; patch_size 2 = 2304 tokens, patch_size 1 = 9216). Cheap (5k train imgs), bidirectional, plays to FN's long-context strength.

(MQAR dropped: synthetic, and the adversarial case for low-rank attention. Kernel correctness is covered by the fidelity tests + the FN-vs-reference parity here.)

**GPU:** kernels are **sm_80+** — use **A100 or L4** (Colab Pro/Pro+). A **T4 will NOT work.**

## 0. Check the GPU

In [ ]:
import torch
name = torch.cuda.get_device_name(); cap = torch.cuda.get_device_capability()
print(name, '| compute capability', cap)
assert cap[0] >= 8, f'Need sm_80+ (A100/L4); this is {name} sm_{cap[0]}{cap[1]}. Switch runtime.'
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## 1. Get the code (clean clone + CUTLASS submodule)

In [ ]:
REPO_URL = 'https://github.com/athrva98/FlashNystrom.git'
# Fresh single clone (no nesting on re-run) WITH submodules

%cd /content
!rm -rf flashnystrom
!git clone --recurse-submodules $REPO_URL flashnystrom
%cd flashnystrom
!git submodule update --init --recursive   # ensure CUTLASS (CuTe) headers
import os
assert os.path.isdir('third_party/cutlass/include'), \
    'CUTLASS submodule did not fetch -- re-run this cell (check network).'
print('CUTLASS headers present - ok to build')

## 2. Build the fused CUDA kernels (~3-6 min)

In [ ]:
import os, torch
cap = torch.cuda.get_device_capability()
os.environ['TORCH_CUDA_ARCH_LIST'] = f'{cap[0]}.{cap[1]}'
# Colab's gcc/CUDA toolchain trips -Werror on third-party headers; LAX disables that guard.
os.environ['FLASH_NYSTROM_LAX_BUILD'] = '1'
!pip install -e . --no-build-isolation

## 3. Verify the kernels

In [ ]:
import torch
from flash_nystrom import flash_nystrom_attention
from flash_nystrom.reference import nystrom_attention_reference
mk = lambda: torch.randn(4, 2, 256, 64, device='cuda', dtype=torch.bfloat16)
q, k, v = mk(), mk(), mk()
o = flash_nystrom_attention(q, k, v, 64, 6); r = nystrom_attention_reference(q, k, v, 64, 6)
print('fwd finite:', bool(torch.isfinite(o).all()), ' max|fn-ref|:', (o.float()-r.float()).abs().max().item())
qg = q.clone().requires_grad_(True); flash_nystrom_attention(qg, k, v, 64, 6).sum().backward()
print('bwd grad finite:', bool(torch.isfinite(qg.grad).all()))

## 4. Mount Drive + output folder
Re-run each session; keep `RUN_NAME` to accumulate, change it for a fresh run.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
RUN_NAME = 'flashnystrom_run'   # change for a separate run
OUTDIR = '/content/drive/MyDrive/flashnystrom_runs/' + RUN_NAME
os.makedirs(OUTDIR + '/figures', exist_ok=True)
print('All results ->', OUTDIR)

## 5. Scaling / crossover  -> `scaling.json` + log
Throughput + peak memory vs N at the auto-found max batch. FN overtakes full attention as N grows, and sdpa OOMs. (~10-20 min)

In [ ]:
cmd = ('python benchmarks/profile_scaling.py'
       ' --backends sdpa flash_nystrom nystrom_reference'
       ' --Ns 256 512 1024 2048 4096 8192 16384'
       f' --json {OUTDIR}/scaling.json 2>&1 | tee {OUTDIR}/scaling.log')
!{cmd}

## 6. STL-10 at 2304 tokens (patch_size 2)  -> `stl10_p2.json` + log
Real 96x96 images, all three backends. FN should match full-attention accuracy at lower memory/step. (~30-60 min; STL-10 auto-downloads ~2.5 GB once.)

In [ ]:
cmd = ('python benchmarks/train_three_way.py'
       ' --dataset stl10 --patch_size 2 --epochs 50 --grad_clip 1.0'
       ' --backends sdpa nystrom_reference flash_nystrom'
       f' 2>&1 | tee {OUTDIR}/stl10_p2.log')
!{cmd}
!cp three_way_results.json {OUTDIR}/stl10_p2.json

## 7. STL-10 at 9216 tokens (patch_size 1) — optional, extreme N
Pixel-level tokens. FN + reference only (full attention OOMs at 9216 tokens — that's the point, and it's already shown in the scaling figure).

In [ ]:
cmd = ('python benchmarks/train_three_way.py'
       ' --dataset stl10 --patch_size 1 --epochs 50 --grad_clip 1.0'
       ' --backends flash_nystrom nystrom_reference'
       f' 2>&1 | tee {OUTDIR}/stl10_p1.log')
!{cmd}
!cp three_way_results.json {OUTDIR}/stl10_p1.json

## 8. Build figures (PDF + PNG) from the saved JSON
Re-run anytime to re-style — reads only the JSON, no GPU.

In [ ]:
cmd = ('python benchmarks/make_figures.py'
       f' --scaling {OUTDIR}/scaling.json'
       f' --cifar {OUTDIR}/stl10_p2.json'
       f' --outdir {OUTDIR}/figures')
!{cmd}

## 9. Inventory + zip (already on Drive)

In [ ]:
import glob, zipfile
arts = sorted(glob.glob(OUTDIR+'/figures/*') + glob.glob(OUTDIR+'/*.json') + glob.glob(OUTDIR+'/*.log'))
print('Saved to Drive:', OUTDIR)
for p in arts: print('  ' + os.path.relpath(p, OUTDIR))
zf = OUTDIR + '/artifacts.zip'
with zipfile.ZipFile(zf, 'w') as z:
    for p in arts: z.write(p, os.path.relpath(p, OUTDIR))
print('zipped ->', zf)

## Notes
- Nothing is lost on disconnect — everything is in `OUTDIR` on Drive.
- Re-style plots with no GPU: re-run section 8.
- STL-10 absolute accuracy is modest (from-scratch ViT on 5k images); the claim is **FN matches full attention + scales better**, not SOTA.
- Preview a figure: `from IPython.display import Image; Image(OUTDIR+'/figures/scaling.png')`.